## ELEX 4653 Quiz 1

Instructions:

- Modify this notebook by adding the Python code described below.

- Test your code using the menu item `Cell  ► Run All` or `Run ► Run All Cells`

- Save the notebook (the .ipynb file) and upload it to the appropriate Assignment folder on the course web site.

### Question 1

Write a function `salestaxes(a)` that returns a two-element tuple, the first element of which is the PST (7%) and the second is the GST (5%) for the amount `a`.

For example, `salestaxes(12.25)` should return the two-element tuple `(0.8575, 0.6125)`.

In [16]:
def salestaxes(a):
    return a*0.07, a*0.05

salestaxes(12.25)

(0.8575, 0.6125)

### Question 2

Write a function `sumabs(l)` that returns the sum of the absolute values of the numbers in the list `l`.

For example, `sumabs([-1,2,-3,5])` should return `11`.

In [17]:
def sumabs(l):
    return sum([abs(x) for x in l])

sumabs([-1,2,-3,5])

11

### Question 3

Write a function `listadd(a,b)` where `a` and `b` are lists of numbers and that returns a list containing the sum of each pair of numbers taken from `a` and `b`.  You can assume both lists have the same number of elements.

For example `listadd([1,2,3],[0.5,0.2,0.2])` would return `[1.5, 2.2, 3.2]`.

In [18]:
def listadd(a,b):
    return [c+d for c,d in zip(a,b)]
    
listadd([1,2,3],[0.5,0.2,0.2])

[1.5, 2.2, 3.2]

In [28]:
# lab validation code; do not modify
def labcheck(testing=0,ntest=10):
    '''
    Python exercise checking.
    Ed.Casas 2023-5-22
    Calls functions q<n>* and checks HMAC of return value[0].
    On mismatch prints return value[1] (function, arguments and return values).
    Setting testing=1 prints HMACs of correct results; paste into 'hashvalues'.
    Note:
    If q<n>* result not JSON-able, convert to string.
    Result order matters for comparison. Sort result if ordering not important.
    '''
    
    import base64, copy, hashlib, json, random, re, string, types 
    from random import randint, uniform
    import traceback
    
    # compare regex to strings that should/shouldn't match
    def checkre(pat,ok,nok):
        for s in ok:
            assert re.search(pat,s), \
                f"pattern '{pat}'\n did not match string '{s}'"
        for s in nok:
            assert not re.search(pat,s), \
                f"pattern '{pat}'\n matched string '{s}'"  
    
    # list of n words with nl letters from chars without repeats
    def randwords(n,chars=string.ascii_lowercase,nl=(2,5)):
        l = []
        while len(l)<n:
            w = ''.join([chars[randint(0,len(chars)-1)] for i in range(randint(*nl))])
            if w not in l:
                l.append(w)
        return l
    
    # convert sets to dicts and dict keys to strings so they can be sorted
    def orderkeys(o):
        if isinstance(o,set):
            return {str(k):None for k in o}
        if isinstance(o,dict):
            return {str(k):orderkeys(v) for k,v in o.items()}
        return o
    
    import math
    def roundn(num, n):
        return 0 if not num else round(num, -int(math.floor(math.log10(abs(num)))) + (n - 1))

    def ch(s,n=(1,1)):
        return randwords(1,chars=s,nl=n)[0]

    def Q1():
        f,a = salestaxes,(round(uniform(1,50),2),)
        oa = copy.deepcopy(a)
        r = repr(f(*a))
        return r,f"{f.__name__}({repr(oa[0])}) returns {r}"   
        
    def Q2():
        f,a = sumabs,([randint(-5,5) for i in range(randint(3,5))],)
        oa = copy.deepcopy(a)
        r = f(*a)
        return r,f"{f.__name__}({repr(oa[0])}) returns {r}"
    
    def Q3():
        f,a = listadd,([randint(-5,5) for i in range(randint(3,5))],
                        [randint(-5,5) for i in range(randint(3,5))])
        oa = copy.deepcopy(a)
        r = f(*a)
        return r,f"{f.__name__}({repr(oa[0])},{repr(oa[1])}) returns {repr(r)}"

    hashvalues = '''
S+hlko1biBBLfVzKDR/GCmmtgcWFtIgrB/M09X7+
CRFbPpKvREknExx2REknXJAnSImaXJAnREknREkn
bRdvuFiuJ4CEM24oyWuhcIz/gPoqD4R81BlY+jw2
'''.split()

    newhash = ''
    dsize = 3 # HMAC base64 digest size (bytes, use 3 or 6 for 4 or 8 char digests)
    dlen = ((dsize*8+5)//6+3)//4*4
           
    for n,f in [(n,f) for n,f in locals().items() if callable(f) and re.search(r'^[Qq]\d+.*',n)]:
        random.seed(n)      
        hashes = '0'*dlen*ntest if testing else hashvalues.pop(0)
        err = ''
        while hashes and not err:
            h, hashes = hashes[:dlen], hashes[dlen:] 
            try:
                v,s = f()
                b = json.dumps(orderkeys(v),sort_keys=True).encode()
                c = base64.b64encode(hashlib.blake2b(b,digest_size=dsize).digest()).decode()
                if testing:
                    print(s)
                    newhash += c
                else:
                    if c != h:
                        err = f"Wrong result for test {n}: {s} (HMAC={c} instead of {h})"
            except Exception as e:
                traceback.print_exc()
                err = f"Error during test {n}: {e}"               
        if testing:
           newhash += '\n'
        else:
            print(err or f"Passed test {n}.")
            
    if testing:
        print(newhash)


labcheck(0)

Passed test Q1.
Passed test Q2.
Passed test Q3.
